# Grand Case Study 1: Banking & Finance (Fraud Detection)

Welcome to the first Grand Case Study. In this notebook, we will synthesize **all** the linear algebra concepts we've learned by solving a real-world banking problem: **Detecting Credit Card Fraud**.

Instead of just defining math, we will ask practical business questions and use Linear Algebra to answer them.

## Question 1: How do we define a customer's transaction profile?
**The Problem:** We need a way for a computer to understand a customer's spending habits.

**The Linear Algebra Answer: Vectors, Norms, and Distances**
We can represent a customer's monthly spending as an $n$-dimensional vector $\mathbf{x}$. Each dimension represents a category (e.g., Groceries, Electronics, Travel). To see if a new transaction is "weird," we can measure the distance (L2 Norm) between the new transaction vector and their historical average vector.

In [ ]:
import numpy as np

# Customer's average monthly spend: [Groceries, Electronics, Travel]
avg_spend = np.array([400, 150, 50])

# A new transaction comes in... something looks suspicious
new_transaction = np.array([450, 2000, 3000])

# Use the L2 Norm (Euclidean Distance) to measure how anomalous this is
distance = np.linalg.norm(avg_spend - new_transaction, ord=2)
print(f"Anomaly Distance Score: {distance:.2f}")
if distance > 1000:
    print("ALERT: Fraud suspected! Distance from normal behavior is too high.")

## Question 2: How do we measure the relationships between different types of transactions across thousands of users?
**The Problem:** If someone spends a lot on "Airlines", do they also spend a lot on "Hotels"? We need to know these relationships across 10,000 customers to flag odd behaviors (e.g., high airline spend but zero hotel spend).

**The Linear Algebra Answer: Matrices and the Covariance Matrix**
We represent our 10,000 customers as a massive $m \times n$ Data Matrix $X$. We can compute the Covariance Matrix $\Sigma = \frac{1}{n-1} X^T X$ to instantly find the correlations between all spending categories.

In [ ]:
np.random.seed(42)
# Simulated matrix: 10,000 customers (rows), 3 categories (cols: Grocery, Airline, Hotel)
# We force Airlines and Hotels to be highly correlated
groceries = np.random.normal(300, 50, 10000)
airlines = np.random.normal(500, 200, 10000)
hotels = airlines * 0.8 + np.random.normal(0, 50, 10000) # Highly correlated to airlines

X = np.column_stack((groceries, airlines, hotels))
X_centered = X - np.mean(X, axis=0)

# Calculate Covariance Matrix using Linear Algebra: X^T * X
covariance_matrix = (X_centered.T @ X_centered) / (X.shape[0] - 1)
print("Covariance Matrix:\n", np.round(covariance_matrix, 2))
print("\nNotice the large positive covariance (~32,000) between Airline (col 1) and Hotel (col 2)!")

## Question 3: How can we identify the "normal" financial behavior patterns?
**The Problem:** Looking at a raw covariance matrix is hard for humans. We want to extract the "primary axes" of normal customer behavior.

**The Linear Algebra Answer: Eigenvalues and Eigenvectors**
By finding the Eigenvectors of the Covariance Matrix, we find the Principal Components. These are the directions in the data where variance is maximized (the "normal" behaviors). The Eigenvalues tell us how important each direction is.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(covariance_matrix)

# Sort them in descending order of importance (highest eigenvalue first)
sort_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sort_idx]
eigenvectors = eigenvectors[:, sort_idx]

print("Eigenvalues (Amount of Variance Explained):\n", np.round(eigenvalues, 2))
print("\nTop Eigenvector (The 'Primary' Behavior Pattern):\n", np.round(eigenvectors[:, 0], 2))
# The top eigenvector heavily weights Airline and Hotel together!

## Question 4: Our transaction dataset is massive. How can we compress it while preserving the important fraud signals?
**The Problem:** We process millions of transactions a minute. Eigendecomposition on the covariance matrix is great, but squaring a massive data matrix $X^T X$ can cause computers to run out of memory or lose numerical precision.

**The Linear Algebra Answer: Singular Value Decomposition (SVD)**
SVD allows us to factorize the raw data matrix directly ($X = U \Sigma V^T$) without ever computing the covariance matrix explicitly. We can then use Truncated SVD to drop the smallest singular values, removing noise and compressing the data.

In [ ]:
# Perform SVD directly on our centered data
U, S, VT = np.linalg.svd(X_centered, full_matrices=False)

print("Singular Values (S):\n", np.round(S, 2))
# Notice that Singular Values squared divided by N-1 equal our Eigenvalues!
print("\nS squared / (N-1):\n", np.round((S**2) / (X.shape[0] - 1), 2))
print("Eigenvalues from Q3:\n", np.round(eigenvalues, 2))

## Question 5: How can we teach an AI to automatically draw a line between normal and fraudulent users?
**The Problem:** We don't want to use manual threshold equations. We want the machine to learn the optimal weights for a mathematical equation $\mathbf{w}^T \mathbf{x} = b$ that separates fraudsters from real customers.

**The Linear Algebra Answer: Systems of Equations, Linear Transformations, and Matrix Calculus**
We treat this as an Overdetermined System of Equations (we have more customers than features). We use Matrix Calculus to find the Gradient of our error (Loss Function) with respect to our weights $\mathbf{w}$. We then update the weights iteratively (Gradient Descent) until the line separates the data perfectly.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Let's pretend the first 100 people in our dataset are fraudsters (Target = 1), rest are normal (0)
labels = np.zeros(10000)
labels[:100] = 1 
X[0:100] = X[0:100] * [1, 5, 0.1] # Make fraudsters behave weirdly (high airline, low hotel)

# Under the hood, this LogisticRegression uses Matrix Calculus (calculating Gradients and Hessians) 
# to optimize the weight vector w.
model = LogisticRegression(solver='lbfgs') # lbfgs is an optimization algorithm relying on the Hessian
model.fit(X, labels)

print("Learned Weight Vector (Linear Transformation for Fraud):", np.round(model.coef_[0], 4))
print("Learned Bias Vector:", np.round(model.intercept_, 4))

# To predict fraud, the model is essentially computing a dot product: w . x + b
sample_user = X[0]
dot_product = np.dot(model.coef_[0], sample_user) + model.intercept_[0]
print(f"\nRaw mathematical output for User 0 (Fraudster): {dot_product:.2f}")
print("(A high positive value means the math flags them as fraud)")